# DA6401 — Continue Training from Existing Checkpoints

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Save & Run All (Commit)**

**What this does:**
Downloads existing `.pth` weights from Google Drive, loads them, and continues
training each model for a few more epochs at a low learning rate.

**Why lower LR for continuation:**
The model is already converged — a high LR would destabilise it.
We use `lr = original_lr / 10` and cosine anneal over the extra epochs.

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available()

In [ ]:
import gdown

# ── Drive file IDs ──────────────────────────────────────────────
# Get these from the share links of each file in your Drive folder
CLASSIFIER_ID = "1jPrAaVV3XYwfuX4y3Z1y2gXnGUYqHXJJ"   # classifier.pth
LOCALIZER_ID  = "11c3ZbZdBN_ztyxOfyQOaWC8Ftt0-XR1j"   # localizer.pth (val IoU=0.683)
UNET_ID       = "1527bugKX9NKd_JAOltXdH_rpU1j6Spzj"   # unet_partial.pth
# ────────────────────────────────────────────────────────────────

downloads = [
    (CLASSIFIER_ID, f"{CKPT}/classifier.pth",  "classifier"),
    (LOCALIZER_ID,  f"{CKPT}/localizer.pth",   "localizer"),
    (UNET_ID,       f"{CKPT}/unet.pth",         "unet"),
]

for drive_id, out_path, label in downloads:
    print(f"Downloading {label}...")
    gdown.download(id=drive_id, output=out_path, quiet=False, fuzzy=True)
    mb = os.path.getsize(out_path) / 1e6 if os.path.exists(out_path) else 0
    print(f"  {'✓' if mb > 0 else '✗'} {label}.pth  ({mb:.0f} MB)\n")

In [ ]:
from data.pets_dataset import download_oxford_pet, create_dataloaders
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

## 1. Continue Localizer Training (+10 epochs)
Best checkpoint: val IoU = 0.683 after 80 epochs.  
Continues at `lr = 1e-4` (10× lower than original).

In [ ]:
import torch, time, numpy as np
import torch.nn as nn
from models.localization import VGG11Localizer
from losses.iou_loss import IoULoss
from train import compute_iou
import wandb

EXTRA_EPOCHS = 10
LR           = 1e-4   # 10x lower than original 1e-3
BATCH_SIZE   = 32
device       = torch.device("cuda")

train_loader, val_loader, _ = create_dataloaders(
    root="./data/oxford_pet", batch_size=BATCH_SIZE, num_workers=2
)

model = VGG11Localizer().to(device)
model.load_state_dict(torch.load(f"{CKPT}/localizer.pth", map_location=device, weights_only=False))
print("✓ Loaded localizer.pth")

# Quick baseline before continuing
model.eval()
iou_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
        iou_sum += ious.sum().item(); total += len(ious)
baseline_iou = iou_sum / total
print(f"  Baseline val IoU: {baseline_iou:.4f}")

wandb.init(project="da6401-assignment2", name="localizer_continued",
           tags=["continue", "localization"],
           config={"extra_epochs": EXTRA_EPOCHS, "lr": LR})

mse_loss  = nn.MSELoss()
iou_loss  = IoULoss(reduction="mean")
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXTRA_EPOCHS, eta_min=1e-6)

best_iou = baseline_iou
for epoch in range(1, EXTRA_EPOCHS + 1):
    model.train()
    t0 = time.time()
    for b in train_loader:
        imgs, bboxes = b["image"].to(device), b["bbox"].to(device)
        optimizer.zero_grad()
        pred = model(imgs)
        (mse_loss(pred, bboxes) + iou_loss(pred, bboxes)).backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    iou_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            ious = compute_iou(model(b["image"].to(device)).cpu(), b["bbox"])
            iou_sum += ious.sum().item(); total += len(ious)
    val_iou = iou_sum / total
    wandb.log({"val/iou": val_iou, "lr": optimizer.param_groups[0]["lr"]})
    print(f"  Epoch {epoch}/{EXTRA_EPOCHS} | Val IoU: {val_iou:.4f} | Time: {time.time()-t0:.0f}s")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), f"{CKPT}/localizer.pth")
        print(f"    ✓ Saved (IoU={val_iou:.4f})")

wandb.finish()
print(f"\nLocalizer: {baseline_iou:.4f} → {best_iou:.4f} (+{best_iou-baseline_iou:.4f})")

## 2. Continue U-Net Segmentation Training (+10 epochs)
Continues from `unet.pth` with partial freeze (blocks 0-2 frozen) at `lr = 5e-4`.

In [ ]:
from models.segmentation import VGG11UNet
from train import compute_dice_score

EXTRA_EPOCHS = 10
LR           = 5e-4

seg_model = VGG11UNet(num_classes=3).to(device)
seg_model.load_state_dict(torch.load(f"{CKPT}/unet.pth", map_location=device, weights_only=False))
print("✓ Loaded unet.pth")

# Keep partial freeze: blocks 0-2 frozen
for name, param in seg_model.encoder.named_parameters():
    param.requires_grad = not any(name.startswith(b) for b in
                                   ["block0", "block1", "block2", "pool0", "pool1", "pool2"])
trainable = sum(p.numel() for p in seg_model.parameters() if p.requires_grad)
print(f"  Trainable params: {trainable:,} (blocks 0-2 frozen)")

# Baseline
seg_model.eval()
dice_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        pred = seg_model(b["image"].to(device)).argmax(1).cpu()
        dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0)
        total += b["image"].size(0)
baseline_dice = dice_sum / total
print(f"  Baseline val Dice: {baseline_dice:.4f}")

wandb.init(project="da6401-assignment2", name="unet_continued",
           tags=["continue", "segmentation"],
           config={"extra_epochs": EXTRA_EPOCHS, "lr": LR})

class_weights = torch.tensor([1.0, 1.0, 2.0]).to(device)
criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = torch.optim.Adam(
    filter(lambda p: p.requires_grad, seg_model.parameters()), lr=LR, weight_decay=1e-4
)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXTRA_EPOCHS, eta_min=1e-6)

best_dice = baseline_dice
for epoch in range(1, EXTRA_EPOCHS + 1):
    seg_model.train()
    t0 = time.time()
    for b in train_loader:
        imgs, masks = b["image"].to(device), b["mask"].to(device)
        optimizer.zero_grad()
        criterion(seg_model(imgs), masks).backward()
        optimizer.step()
    scheduler.step()

    seg_model.eval()
    dice_sum, total = 0.0, 0
    with torch.no_grad():
        for b in val_loader:
            pred = seg_model(b["image"].to(device)).argmax(1).cpu()
            dice_sum += compute_dice_score(pred, b["mask"]) * b["image"].size(0)
            total += b["image"].size(0)
    val_dice = dice_sum / total
    wandb.log({"val/dice": val_dice, "lr": optimizer.param_groups[0]["lr"]})
    print(f"  Epoch {epoch}/{EXTRA_EPOCHS} | Val Dice: {val_dice:.4f} | Time: {time.time()-t0:.0f}s")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(seg_model.state_dict(), f"{CKPT}/unet.pth")
        print(f"    ✓ Saved (Dice={val_dice:.4f})")

wandb.finish()
print(f"\nU-Net: {baseline_dice:.4f} → {best_dice:.4f} (+{best_dice-baseline_dice:.4f})")

## 3. Final Summary

In [ ]:
print("="*60)
print("  FILES TO DOWNLOAD FROM OUTPUT TAB")
print("="*60)
for f in sorted(os.listdir(CKPT)):
    mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"  {f:40s}  {mb:.0f} MB")
print()
print("  NEXT STEPS:")
print("  1. Download localizer.pth and unet.pth from Output tab")
print("  2. Upload both to Google Drive (replace old files or add new ones)")
print("  3. Share each file → get file ID from the link")
print("  4. Paste the new file IDs → update models/multitask.py")
print("  5. Rebuild ZIP and resubmit to Gradescope")
print("="*60)